# Task 2 · Contracts, nested data and a star schema

**Name:** Hanna Mounir  
**ID:** 58-11495 · **Group:** IET-01

The original input file stays unchanged.

In [29]:
DATA = "bookings.jsonl"

## Part A · The contract 

Use Pydantic to check each booking, then keep only the first **valid** copy of each `booking_id`.

In [30]:
import json
import shutil
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field, ValidationError, model_validator

pd.set_option("display.max_columns", 30)

In [31]:
class Ticket(BaseModel):
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: date
    venue: str


class Booking(BaseModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: str | None = None  # new field; older bookings can omit it

    @model_validator(mode="after")
    def check_total_and_date(self):
        ticket_total = sum(t.qty * t.price for t in self.tickets)
        if abs(self.total - ticket_total) > 0.01:
            raise ValueError("total mismatch")

        booked_date = self.booked_at.astimezone(timezone.utc).date()
        if booked_date >= self.event.date:
            raise ValueError("booking on or after event date")
        return self

In [32]:
with open(DATA, encoding="utf-8") as f:
    lines = f.readlines()

valid = []
quarantine = []
seen = set()
raw_documents = []

for line_number, line in enumerate(lines, 1):
    try:
        record = json.loads(line)
    except json.JSONDecodeError:
        quarantine.append({"booking_id": None, "reason": "invalid JSON"})
        continue

    if isinstance(record, dict):
        raw_documents.append(record)

    try:
        booking = Booking.model_validate(record)
    except ValidationError as e:
        error = e.errors()[0]  # first failing rule for this booking
        reason = ".".join(str(p) for p in error["loc"] if not isinstance(p, int)) or error["msg"].removeprefix("Value error, ")
        booking_id = record.get("booking_id") if isinstance(record, dict) else None
        quarantine.append({"booking_id": booking_id, "reason": reason})
        continue

    if booking.booking_id in seen:
        quarantine.append({"booking_id": booking.booking_id, "reason": "duplicate booking_id"})
        continue

    seen.add(booking.booking_id)
    valid.append(booking)

with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for row in quarantine:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [33]:
# ✓ check
assert len(lines) == len(valid) + len(quarantine)
print("Lines read:", len(lines))
print("Valid bookings:", len(valid))
print("Quarantined:", len(quarantine))

reason_counts = Counter(row["reason"] for row in quarantine)
pd.DataFrame(reason_counts.items(), columns=["reason", "count"]).sort_values("reason")

Lines read: 283
Valid bookings: 259
Quarantined: 24


,reason,count
6,booking on or after event date,3
3,duplicate booking_id,3
4,event,2
2,student.id,4
7,tickets,2
1,tickets.qty,3
0,tickets.type,3
5,total mismatch,4


## Part B · Flatten 

One row per booking in `bookings`; one row per ticket entry in `booking_tickets`.

In [34]:
booking_rows = []
ticket_rows = []

for b in valid:
    booking_rows.append({
        "booking_id": b.booking_id,
        "booked_at": b.booked_at,
        "student_id": b.student.id,
        "student_name": b.student.name,
        "student_faculty": b.student.faculty,
        "event_id": b.event.id,
        "event_title": b.event.title,
        "event_club": b.event.club,
        "event_date": b.event.date,
        "event_venue": b.event.venue,
        "total": b.total,
        "payment": b.payment,
        "status": b.status,
        "promo_code": b.promo_code,
    })

    for t in b.tickets:
        ticket_rows.append({
            "booking_id": b.booking_id,
            "type": t.type,
            "qty": t.qty,
            "price": t.price,
            "amount": t.qty * t.price,
        })

bookings = pd.DataFrame(booking_rows, columns=[
    "booking_id", "booked_at", "student_id", "student_name", "student_faculty",
    "event_id", "event_title", "event_club", "event_date", "event_venue",
    "total", "payment", "status", "promo_code",
])
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True)

booking_tickets = pd.DataFrame(ticket_rows, columns=[
    "booking_id", "type", "qty", "price", "amount",
])

print("Bookings:", bookings.shape)
print("Booking tickets:", booking_tickets.shape)
bookings.head(3)

Bookings: (259, 14)
Booking tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,None
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,None
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,None


In [35]:
booking_tickets.head(3)

,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0


In [36]:
# ✓ check
assert bookings["booking_id"].is_unique
per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
assert (bookings.set_index("booking_id")["total"].sub(per_booking).abs() < 0.01).all()
print("✓ booking_id is unique; ticket amounts match booking totals")

✓ booking_id is unique; ticket amounts match booking totals


### Schema drift

Find the new top-level field and the first booking date when it appears. Missing `promo_code` becomes `None` through the Pydantic default.

In [37]:
example_fields = {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"}
new_fields = sorted(set().union(*(set(r) for r in raw_documents)) - example_fields)

for field in new_fields:
    dates = pd.to_datetime(
        [r.get("booked_at") for r in raw_documents if field in r],
        errors="coerce", utc=True,
    )
    first_date = dates.min()
    print(field, "first appears:", first_date.date() if pd.notna(first_date) else "unknown")

promo_code first appears: 2026-10-20


## Part C · Model and store (Lab 5)

DuckDB stores two dimension tables and a fact table. **One fact row = one ticket entry.**

In [38]:
import duckdb

con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
con.register("bookings_input", bookings)
con.register("tickets_input", booking_tickets)
con.sql("create or replace table bookings as select * from bookings_input")
con.sql("create or replace table booking_tickets as select * from tickets_input")

In [39]:
con.sql("""
    create or replace table dim_event as
    select row_number() over (order by event_id) as event_key,
           event_id, event_title, event_club, event_date, event_venue
    from (
        select distinct on (event_id)
               event_id, event_title, event_club, event_date, event_venue
        from bookings
        order by event_id, booked_at
    )
""")

con.sql("""
    create or replace table dim_student as
    select row_number() over (order by student_id) as student_key,
           student_id, student_name, student_faculty
    from (
        select distinct on (student_id)
               student_id, student_name, student_faculty
        from bookings
        order by student_id, booked_at
    )
""")

con.sql("""
    create or replace table fact_ticket_sales as
    select t.booking_id, e.event_key, s.student_key,
           t.type as ticket_type, t.qty, t.price, t.amount, b.status,
           strftime(b.booked_at, '%Y%m%d')::integer as date_key,
           strftime(b.booked_at, '%Y-%m') as booked_month
    from booking_tickets t
    join bookings b on b.booking_id = t.booking_id
    join dim_event e on e.event_id = b.event_id
    join dim_student s on s.student_id = b.student_id
""")

con.sql("select * from fact_ticket_sales limit 5").df()

,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,date_key,booked_month
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928,2026-09
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928,2026-09
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927,2026-09
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013,2026-10
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101,2026-11


In [40]:
# ✓ check
fact_rows = con.sql("select count(*) from fact_ticket_sales").fetchone()[0]
fact_total = con.sql("select coalesce(sum(amount), 0) from fact_ticket_sales").fetchone()[0]

assert fact_rows == len(booking_tickets)
assert abs(fact_total - bookings["total"].sum()) < 0.01
assert con.sql("select count(*) from dim_event").fetchone()[0] == bookings["event_id"].nunique()
assert con.sql("select count(*) from dim_student").fetchone()[0] == bookings["student_id"].nunique()
print("✓", fact_rows, "fact rows; total amount =", fact_total)

✓ 314 fact rows; total amount = 31285.0


In [41]:
# Confirmed bookings only: tickets sold and revenue by club and ticket type
club_sales = con.sql("""
    select e.event_club as club, f.ticket_type,
           sum(f.qty) as tickets_sold,
           round(sum(f.amount), 2) as revenue
    from fact_ticket_sales f
    join dim_event e on e.event_key = f.event_key
    where f.status = 'confirmed'
    group by e.event_club, f.ticket_type
    order by e.event_club, f.ticket_type
""").df()
club_sales

,club,ticket_type,tickets_sold,revenue
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


In [42]:
# Write the two dimensions as files and partition the fact by booking month
STAR = Path("star")
shutil.rmtree(STAR, ignore_errors=True)
STAR.mkdir()

con.sql("copy dim_event to 'star/dim_event.parquet' (format parquet)")
con.sql("copy dim_student to 'star/dim_student.parquet' (format parquet)")
con.sql("""
    copy fact_ticket_sales to 'star/fact_ticket_sales'
    (format parquet, partition_by (booked_month))
""")

for path in sorted(STAR.rglob("*.parquet")):
    print(path)

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet


## Content

- **Part A:** a record-level Pydantic contract and JSONL quarantine, with duplicate checks after validation.
- **Part B:** two flat tables at different grains and a schema-drift check.
- **Part C:** DuckDB dimensions, ticket-level fact, assertions, confirmed-only sales query, and partitioned Parquet export.

